# Notebook 1 — Losses, likelihood, and conditioning

**Week 2 · Day 1 · ≈ 30 min, after Labwork 1**

> This is where you *look* at what you built. Every figure below is drawn from **your**
> `optlab` code — if a plot is wrong, the implementation behind it is wrong.

Read Lecture 1 first and finish Labwork 1. This notebook needs, from day 1:

| From | What |
|---|---|
| `numerics/gradcheck.py` | `numerical_gradient`, `check_gradient` |
| `losses.py` | `SquaredError`, `LogisticNLL` |
| `problems/glm.py` | `GLMLoss.value`, `GLMLoss.gradient`, `linear_regression`, `logistic_regression` |
| `problems/quadratic.py` | `Quadratic`, `Quadratic.ill_conditioned` |
| `problems/rosenbrock.py` | `Rosenbrock.value`, `Rosenbrock.gradient` |
| `datasets/load.py` | `standardize` |

Five figures, in the order the lecture made the claims:

1. the likelihood and the loss are the same picture upside down;
2. what the two losses actually look like, and why one of them cannot be computed naively;
3. level sets against the condition number $\kappa$;
4. $\kappa$ is decided by your data, not by your algorithm;
5. the finite-difference error U-curve.

Then the part that is not optional: **breaking** the gradient checker, to see what it
does *not* catch.

## 0. Setup

`scipy`, `scikit-learn` and `matplotlib` are all allowed here — a notebook is outside
`src/`, so it may use oracles. Inside the package, still numpy only.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# The optlab root is the nearest ancestor holding pyproject.toml, so this works whether
# Jupyter was started in notebooks/ or in the repository root.
HERE = Path.cwd()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "pyproject.toml").exists()), HERE.parent)

# Prefer the installed package (`make install`); fall back to <root>/src if it is not.
try:
    import optlab
except ModuleNotFoundError:
    sys.path.insert(0, str(ROOT / "src"))
    import optlab

sys.path.insert(0, str(ROOT))  # for datasets/

from datasets.load import standardize                                        # noqa: E402
from optlab.losses import LogisticNLL, SquaredError                          # noqa: E402
from optlab.numerics import check_gradient, numerical_gradient               # noqa: E402
from optlab.problems import (                                                # noqa: E402
    Quadratic,
    Rosenbrock,
    linear_regression,
    logistic_regression,
)

np.set_printoptions(precision=6, suppress=True)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
                     "axes.titlesize": 10, "figure.dpi": 110})
rng = np.random.default_rng(20250921)

print("optlab root  :", ROOT)
print("optlab loaded:", Path(optlab.__file__).parent)

Before anything is plotted, find out what is actually implemented. A missing piece here
is not a bug in the notebook — it is a stub in your package that Labwork 1 asked you to
fill in.

In [ ]:
def status(name, thunk):
    try:
        thunk()
    except NotImplementedError:
        return f"  MISSING   {name}"
    except Exception as err:                      # noqa: BLE001 - we want to see anything
        return f"  BROKEN    {name}   ({type(err).__name__}: {err})"
    return f"  ok        {name}"


v, Xh, yh = np.array([1.0, 2.0]), np.array([[1.0], [2.0]]), np.array([2.0, 4.0])

needed = [
    ("numerical_gradient",        lambda: numerical_gradient(lambda u: float(u @ u), v)),
    ("check_gradient",            lambda: check_gradient(lambda u: float(u @ u), lambda u: 2 * u, v)),
    ("SquaredError",              lambda: SquaredError().value(v, v)),
    ("LogisticNLL",               lambda: LogisticNLL().value(v, np.zeros(2))),
    ("GLMLoss.value / .gradient", lambda: linear_regression(Xh, yh).gradient(np.zeros(1))),
    ("Quadratic.ill_conditioned", lambda: Quadratic.ill_conditioned(2, 10.0).value(v)),
    ("Rosenbrock",                lambda: Rosenbrock().gradient(v)),
    ("standardize",               lambda: standardize(np.array([[0.0], [1.0], [2.0]]))),
]

print("What this notebook needs from day 1:\n")
print("\n".join(status(name, thunk) for name, thunk in needed))

---

## 1. The likelihood and the loss are one picture

The lecture's headline was an identity, not an analogy:

$$\hat w = \arg\max_w \; \prod_{i=1}^{n} p(y_i \mid x_i, w) \qquad=\qquad \arg\min_w \; -\frac{1}{n}\sum_{i=1}^{n} \log p(y_i \mid x_i, w)$$

The product on the left is a probability: it asks *how plausible is this data, if the
parameter were $w$?* The sum on the right is what you minimize. They are the same
statement — $\log$ is increasing, and negating turns the peak into a valley.

One parameter is enough to see it, so take a model with a single coefficient. With
Gaussian noise of variance $\sigma^2$ the two are linked exactly:

$$\prod_i p(y_i \mid w) \;\propto\; \exp\!\left(-\frac{n\,L(w)}{\sigma^{2}}\right),
\qquad L(w) = \frac{1}{n}\sum_i \tfrac12 (z_i - y_i)^2$$

and $L$ is precisely what your `linear_regression(X, y).value` returns.

In [ ]:
n, w_true, sigma = 12, 2.0, 1.0
X = rng.normal(size=(n, 1))
y = X[:, 0] * w_true + sigma * rng.normal(size=n)

loss = linear_regression(X, y)                      # your GLMLoss + SquaredError

grid = np.linspace(w_true - 3.0, w_true + 3.0, 601)
L = np.array([loss.value(np.array([w])) for w in grid])
likelihood = np.exp(-n * L / sigma**2)
likelihood /= likelihood.max()                      # only the shape matters

w_argmax = grid[likelihood.argmax()]
w_argmin = grid[L.argmin()]
w_normal = np.linalg.solve(X.T @ X, X.T @ y)[0]     # the closed form, as an oracle

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(grid, likelihood, lw=2, color="tab:blue")
ax[0].axvline(w_argmax, color="tab:red", ls="--", lw=1.2)
ax[0].set_title("likelihood  $\\prod_i p(y_i \\mid w)$   — MAXIMIZE")
ax[0].set_xlabel("$w$"), ax[0].set_ylabel("relative plausibility")

ax[1].plot(grid, L, lw=2, color="tab:green")
ax[1].axvline(w_argmin, color="tab:red", ls="--", lw=1.2)
ax[1].set_title("loss  $L(w) = \\frac{1}{n}\\sum_i \\frac{1}{2} (z_i - y_i)^2$   — MINIMIZE")
ax[1].set_xlabel("$w$"), ax[1].set_ylabel("loss")
fig.tight_layout()
plt.show()

print(f"argmax of the likelihood : {w_argmax:.6f}")
print(f"argmin of the loss       : {w_argmin:.6f}")
print(f"normal equations (oracle): {w_normal:.6f}")
print(f"true w used to generate  : {w_true:.6f}   (n = {n}, so w-hat is near it, not on it)")

**Figure 1 — the same estimate, from both directions.**

The peak on the left and the valley on the right sit at the same $w$, and both agree with
the closed-form solution of the normal equations to the resolution of the grid. Nothing
was fitted twice: the right-hand curve *is* the left-hand one, log-transformed and
flipped.

Two details worth noticing, because they come back all week:

- The minimizer is not $w_{\text{true}}$. With $n = 12$ noisy observations it cannot be.
  **Optimization finds the minimum of the loss you wrote; whether that is the answer you
  wanted is a modelling question, not an optimization one.** This week we are strict
  about the first and honest about the second.
- The valley is smooth and has exactly one bottom. That is convexity, and it is why
  every method in this course works on this problem. Day 4's Rosenbrock and day 6's
  lasso are where that stops being free.

---

## 2. What the two losses actually look like

Both losses are functions of the **linear predictor** $z_i = x_i^\top w$, which is the
whole reason one `GLMLoss` class can serve every likelihood: change $\varphi$, keep
everything else. So plot $\varphi$ directly, against $z$.

$$\varphi_{\text{sq}}(z, y) = \tfrac12 (z-y)^2, \qquad
  \varphi_{\text{log}}(z, y) = \log(1 + e^{z}) - y\,z$$

In [ ]:
z = np.linspace(-6.0, 6.0, 601)
ones, zeros = np.ones_like(z), np.zeros_like(z)
sq, lg = SquaredError(), LogisticNLL()

curves = [
    ("squared error, $y = 1$", sq.value(z, ones), sq.d1(z, ones), "tab:green"),
    ("logistic, $y = 1$",      lg.value(z, ones), lg.d1(z, ones), "tab:blue"),
    ("logistic, $y = 0$",      lg.value(z, zeros), lg.d1(z, zeros), "tab:orange"),
]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for label, val, d1, colour in curves:
    ax[0].plot(z, val, lw=2, color=colour, label=label)
    ax[1].plot(z, d1, lw=2, color=colour, label=label)
ax[0].set_title("$\\varphi(z, y)$ — the loss"), ax[0].set_ylim(-0.3, 8)
ax[1].set_title("$\\partial\\varphi/\\partial z$ — what the gradient is built from")
for a in ax:
    a.set_xlabel("linear predictor $z$"), a.axhline(0, color="k", lw=0.8), a.legend(fontsize=8)
fig.tight_layout()
plt.show()

print("range of d1 over z in [-6, 6]")
for label, _, d1, _ in curves:
    print(f"  {label:24s} min {d1.min():9.4f}   max {d1.max():9.4f}")

**Figure 2 — one of these is bounded and one is not.**

*Left.* Squared error is a parabola: being wrong by $6$ costs $18$, and being wrong by
$60$ costs $1800$. The logistic loss is asymptotically **linear** on the wrong side and
flattens to zero on the right side — a confidently correct prediction contributes
essentially nothing more, however confident it becomes.

*Right.* This is the panel that matters for optimization, because
$\nabla L(w) = \frac{1}{n} X^\top \varphi'(Xw, y)$ — the gradient is $X^\top$ times
*this* curve. Read the printed ranges: the logistic derivative is $\sigma(z) - y$, which
is trapped in $(-1, 1)$ no matter how wrong the prediction is, while the squared-error
derivative $z - y$ grows without bound.

That single fact explains a lot of day 2 and day 6: an outlier under squared error can
dominate the whole gradient, and the day-6 Huber loss is built precisely to bound it.
Note also that logistic $y=0$ and $y=1$ are mirror images — as they must be, since
swapping the label swaps the two classes.

### The trap: the same formula, computed badly

$\log(1 + e^{z})$ is fine on paper and a minefield in float64. Your `LogisticNLL`
should already survive this; here is the proof, next to the naive version.

In [ ]:
print(f"{'z':>8}  {'naive log(1+exp(z))':>22}  {'your LogisticNLL':>18}   verdict")
for zz in [1.0, 30.0, 500.0, 700.0, 710.0, 1000.0]:
    with np.errstate(over="ignore"):
        naive = float(np.log(1.0 + np.exp(zz)))
    yours = float(lg.value(np.array([zz]), np.array([0.0]))[0])
    if np.isfinite(naive):
        verdict = "agree" if abs(naive - yours) <= 1e-12 * max(1.0, abs(yours)) else "DISAGREE"
    else:
        verdict = "naive overflowed; yours did not"
    print(f"{zz:8.1f}  {naive:>22.6f}  {yours:>18.6f}   {verdict}")

print("\nWhy: exp(710) is larger than the largest float64.")
print("  largest float64 :", np.finfo(np.float64).max)
with np.errstate(over="ignore"):
    print("  exp(700)        :", float(np.exp(700.0)))
    print("  exp(710)        :", float(np.exp(710.0)))

The failure is not gradual. Up to $z = 700$ the naive formula is perfect; one step
further it is `inf`, and `inf` propagates through the rest of the computation as a
silent, total loss of information — the gradient becomes `nan`, the optimizer takes a
`nan` step, and every subsequent iterate is `nan`.

The fix is the identity

$$\log(1+e^{z}) = \max(z, 0) + \log\!\left(1 + e^{-|z|}\right)$$

which never exponentiates a positive number. It is the same function, written so that
the intermediate quantities stay in range. **This is the whole of numerical analysis in
one line:** mathematically equal expressions are not computationally equal, and choosing
between them is part of writing the code.

---

## 3. Level sets against the condition number

For $f(x) = \tfrac12 x^\top A x$ with $A$ symmetric positive definite, the level sets
$\{x : f(x) = c\}$ are ellipses whose axes are the eigenvectors of $A$. The
**condition number**

$$\kappa(A) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

is the one number that says how hard this problem is for a first-order method. Your
`Quadratic.ill_conditioned(n, kappa)` builds an instance with exactly that $\kappa$, so
we can dial the difficulty and watch the geometry respond.

In [ ]:
def contour_grid(q, lim, m=141):
    """Evaluate YOUR Quadratic.value on a grid — no shortcut through q.A."""
    g = np.linspace(-lim, lim, m)
    G1, G2 = np.meshgrid(g, g)
    Z = np.array([[q.value(np.array([a, b])) for a in g] for b in g])
    return G1, G2, Z


def measured_axis_ratio(q, level=0.5):
    """Half-lengths of the level set along each axis, from YOUR Quadratic.value.

    Along axis i the function is quadratic, so f(t e_i) = t^2 f(e_i); setting that
    equal to `level` gives t_i = sqrt(level / f(e_i)) with a single evaluation.
    """
    t = [np.sqrt(level / q.value(e)) for e in np.eye(2)]
    return max(t) / min(t)


kappas = [1.0, 10.0, 100.0]
fig, ax = plt.subplots(1, 3, figsize=(12, 3.9))

for a, kappa in zip(ax, kappas, strict=True):
    q = Quadratic.ill_conditioned(2, kappa)
    G1, G2, Z = contour_grid(q, 1.5)
    a.contour(G1, G2, Z, levels=np.linspace(0.05, 1.2, 9), linewidths=1.1)
    a.plot(0, 0, "r*", ms=13)

    x = np.array([1.2, 1.2 / np.sqrt(kappa)])          # a point on a level set
    g = q.gradient(x)
    # The blue arrow is drawn longer than the red one so that it is still visible in the
    # kappa = 1 panel, where the two directions coincide exactly.
    a.arrow(*x, *(-0.85 * x / np.linalg.norm(x)), color="tab:blue",
            width=0.02, length_includes_head=True, zorder=4)
    a.arrow(*x, *(-0.6 * g / np.linalg.norm(g)), color="tab:red",
            width=0.02, length_includes_head=True, zorder=5)

    lam = np.linalg.eigvalsh(q.A)
    a.set_title(f"$\\kappa$ = {kappa:g}   (eigenvalues {lam[0]:.3g}, {lam[1]:.3g})")
    a.set_aspect("equal"), a.set_xlim(-1.5, 1.5), a.set_ylim(-1.5, 1.5)

ax[0].legend(handles=[plt.Line2D([], [], color="tab:red", lw=2, label="$-\\nabla f$"),
                      plt.Line2D([], [], color="tab:blue", lw=2, label="towards the minimum")],
             fontsize=8, loc="lower left")
fig.tight_layout()
plt.show()

print(f"{'kappa':>8}  {'measured ratio':>15}  {'sqrt(kappa)':>12}  "
      f"{'angle(-grad, -x)':>18}  {'(k-1)/(k+1)':>12}")
for kappa in kappas:
    q = Quadratic.ill_conditioned(2, kappa)
    x = np.array([1.2, 1.2 / np.sqrt(kappa)])
    g = q.gradient(x)
    cos = float(-g @ -x / (np.linalg.norm(g) * np.linalg.norm(x)))
    angle = np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))
    print(f"{kappa:8g}  {measured_axis_ratio(q):15.4f}  {np.sqrt(kappa):12.4f}  "
          f"{angle:17.2f}°  {(kappa - 1) / (kappa + 1):12.4f}")

**Figure 3 — what $\kappa$ does to the geometry, and to the gradient.**

Three facts, all of them in the printed table:

1. **The ellipses are $\sqrt{\kappa}$ times longer than they are wide**, not $\kappa$
   times — a distinction worth getting right, and one the table *measures* rather than
   assumes: `measured_axis_ratio` finds the half-length of a level set along each axis
   by evaluating your `Quadratic.value`, and the column beside it is $\sqrt{\kappa}$ for
   comparison. The reason is that the level set $\tfrac12\lambda_i x_i^2 = c$ reaches
   $x_i = \sqrt{2c/\lambda_i}$, so the semi-axes scale like $1/\sqrt{\lambda_i}$.
   $\kappa = 100$ gives a valley ten times longer than it is wide.
2. **The steepest-descent direction is not the direction of the minimum.** The red arrow
   is $-\nabla f$, the blue one points at the star. They coincide only in the circular
   case $\kappa = 1$ — in that first panel the blue arrow is drawn longer purely so you
   can see it is there, underneath the red one. The angle between them in the printed
   table grows towards $90°$ as $\kappa$ grows. Gradient descent will therefore cross the valley instead of running
   down it — that is the zigzag you will draw tomorrow.
3. **$(\kappa-1)/(\kappa+1)$ is the price.** That is the factor by which the error of
   gradient descent with an exact line search is multiplied per iteration on this
   problem. At $\kappa = 100$ it is $0.9802$, so reducing the error by $10^{-6}$ needs
   roughly $\log(10^{-6})/\log(0.9802) \approx 690$ iterations. At $\kappa = 1$ it is
   zero — one step.

Momentum improves the factor to $(\sqrt{\kappa}-1)/(\sqrt{\kappa}+1)$ (day 2), and Newton
removes it entirely by rescaling the space so the ellipses become circles (day 4). Those
are the two *algorithmic* responses to a bad $\kappa$. The third is not algorithmic at
all, and it is the subject of the next section: do not create it in the first place.

---

## 4. $\kappa$ is decided by your data

The quadratic above had its condition number dialled in by hand. Real problems get theirs
from the data: for a linear model the relevant matrix is $A = X^\top X / n$, and $X$ is
whatever your columns happen to be.

Here are two features that carry the **same information** — they are mildly correlated
and equally informative — differing only in that the second was recorded in units a
thousand times smaller. Millimetres instead of metres.

In [ ]:
m = 500
f1 = rng.normal(size=m)
f2 = 0.3 * f1 + rng.normal(size=m)
X_raw = np.column_stack([f1, 1e-3 * f2])            # second column in "millimetres"
X_std = standardize(X_raw)                          # your datasets/load.standardize

A_raw = X_raw.T @ X_raw / m
A_std = X_std.T @ X_std / m

fig, ax = plt.subplots(1, 2, figsize=(11, 4.0))
panels = [(A_raw, "raw features"), (A_std, "after standardize()")]
for a, (A, title) in zip(ax, panels, strict=True):
    q = Quadratic(A, np.zeros(2))
    g = np.linspace(-2.5, 2.5, 141)
    Z = np.array([[q.value(np.array([p, r])) for p in g] for r in g])
    a.contour(*np.meshgrid(g, g), Z, levels=np.geomspace(Z[Z > 0].min() + 1e-12, Z.max(), 12),
              linewidths=1.1)
    a.plot(0, 0, "r*", ms=13)
    a.set_aspect("equal")
    a.set_title(f"{title}   —   $\\kappa$ = {np.linalg.cond(A):.4g}")
    a.set_xlabel("$w_1$"), a.set_ylabel("$w_2$")
fig.tight_layout()
plt.show()

for name, A in [("raw         ", A_raw), ("standardized", A_std)]:
    lam = np.linalg.eigvalsh(A)
    kappa = lam.max() / lam.min()
    print(f"{name}  eigenvalues {lam[0]:12.6g} {lam[1]:12.6g}   kappa = {kappa:12.6g}"
          f"   ellipse is {np.sqrt(kappa):.4g}x longer than wide")
print(f"\nkappa fell by a factor of {np.linalg.cond(A_raw) / np.linalg.cond(A_std):.4g}, "
      "and not one number in the data changed meaning.")

**Figure 4 — the same problem, in two coordinate systems.**

On the left the level sets are not ellipses you can see: on this window they are
near-parallel stripes, because — as the printed line says — the valley is about a
thousand times longer than it is wide. On the right they are the mild ellipses of
$\kappa \approx 1.9$. Gradient descent on the left-hand picture is hopeless; on the
right-hand one it is easy. **The two panels are the same data and the same model.** Only
the units changed.

So standardizing is not statistical etiquette, it is an optimization decision — and it is
free. It is also the first thing to check when an optimizer refuses to converge on real
data.

The honest caveat: standardizing changes what $w$ *means*. The coefficients now speak in
standard deviations, so convert back before interpreting them, and remember that the
penalty $\lambda\|w\|^2$ of day 4 is not invariant to this rescaling either — which is
exactly why ridge regression is almost always applied to standardized features.

In [ ]:
# Optional: the same measurement on real data, if the instructor's data/ folder is present.
try:
    from datasets.load import load

    X_cal, _ = load("california")
    k_raw = np.linalg.cond(X_cal.T @ X_cal / len(X_cal))
    Xs = standardize(X_cal)
    k_std = np.linalg.cond(Xs.T @ Xs / len(Xs))
    print(f"California housing, {X_cal.shape[0]} x {X_cal.shape[1]}")
    print(f"  kappa raw          {k_raw:12.6g}")
    print(f"  kappa standardized {k_std:12.6g}")
except (FileNotFoundError, ModuleNotFoundError) as err:
    print("Real datasets not available in this clone -- the synthetic example above makes")
    print("the same point. To enable this cell, the instructor runs datasets/prepare.py.")
    print(f"  ({type(err).__name__}: {err})")

---

## 5. The finite-difference U-curve

A central difference approximates $\partial f/\partial x_i$ by

$$\frac{f(x + h e_i) - f(x - h e_i)}{2h}$$

and its error is a fight between two terms that pull in opposite directions:

| term | size | wants |
|---|---|---|
| truncation (Taylor remainder) | $O(h^2)$ | $h$ **small** |
| rounding (cancellation in the subtraction) | $O(\varepsilon / h)$ | $h$ **large** |

Minimizing $c h^2 + \varepsilon/h$ gives $h^\star \propto \varepsilon^{1/3} \approx 6\cdot10^{-6}$
for float64 — which is where the default `h = 1e-6` in your `numerical_gradient` comes
from. A forward difference has an $O(h)$ truncation term instead, so its optimum sits at
$\varepsilon^{1/2} \approx 1.5\cdot10^{-8}$ and it never gets as accurate.

Rosenbrock's gradient is analytic and correct (Labwork 1 checked it), so we can use it as
the truth and measure the error of the approximation.

In [ ]:
rosen = Rosenbrock()
x0 = np.array([-1.2, 1.0])                 # the classic hard starting point
g_true = rosen.gradient(x0)
basis = np.eye(2)

hs = np.logspace(-14.0, -1.0, 66)
err_central, err_forward = [], []
for h in hs:
    gc = numerical_gradient(rosen.value, x0, h)                       # yours
    gf = np.array([(rosen.value(x0 + h * e) - rosen.value(x0)) / h for e in basis])
    err_central.append(np.linalg.norm(gc - g_true) / np.linalg.norm(g_true))
    err_forward.append(np.linalg.norm(gf - g_true) / np.linalg.norm(g_true))

err_central, err_forward = np.array(err_central), np.array(err_forward)
eps = np.finfo(np.float64).eps

# Slope guides, anchored to the measured branches rather than to invented constants:
# each one is pinned at one end of the curve it explains, so only the SLOPE is a claim.
guide_h2 = err_central[-1] * (hs / hs[-1]) ** 2       # truncation, central
guide_h1 = err_forward[-1] * (hs / hs[-1])            # truncation, forward
guide_eps = err_central[0] * (hs[0] / hs)             # rounding, both

plt.figure(figsize=(6.8, 4.3))
plt.loglog(hs, err_central, "o-", ms=3, lw=1.5, label="central  (yours)")
plt.loglog(hs, err_forward, "s-", ms=3, lw=1.5, label="forward")
plt.loglog(hs, guide_h2, "k--", lw=1, label="slope $+2$: $O(h^2)$ truncation")
plt.loglog(hs, guide_h1, "k:", lw=1, label="slope $+1$: $O(h)$ truncation")
plt.loglog(hs, guide_eps, "r--", lw=1, label="slope $-1$: $O(\\varepsilon/h)$ rounding")
plt.axvline(eps ** (1 / 3), color="tab:blue", lw=1, alpha=0.6)
plt.axvline(eps ** (1 / 2), color="tab:orange", lw=1, alpha=0.6)
plt.ylim(1e-13, 1e2), plt.xlabel("step $h$"), plt.ylabel("relative error of the gradient")
plt.title("finite differences: too small is as bad as too large")
plt.legend(fontsize=8, loc="lower left", framealpha=0.92), plt.tight_layout()
plt.show()

print(f"machine epsilon             {eps:.4e}")
print(f"central: best h = {hs[err_central.argmin()]:.3e}   error {err_central.min():.3e}"
      f"   (theory: h* ~ eps^(1/3) = {eps ** (1/3):.3e})")
print(f"forward: best h = {hs[err_forward.argmin()]:.3e}   error {err_forward.min():.3e}"
      f"   (theory: h* ~ eps^(1/2) = {eps ** (1/2):.3e})")
print(f"at the default h = 1e-6, central gives {np.interp(1e-6, hs, err_central):.3e}")

**Figure 5 — the U, and why the default is $10^{-6}$.**

Read it from the right. As $h$ shrinks the error falls along a slope of $+2$, a hundred
times better per decade — then it turns around and climbs along a slope of $-1$. The
three straight guides are pinned to one end of the branch they explain, so only their
*slope* is a claim: $O(h^2)$ and $O(h)$ truncation on the right, $O(\varepsilon/h)$
rounding on the left. The two vertical lines are the predicted optima
$\varepsilon^{1/3}$ and $\varepsilon^{1/2}$; the measured minima printed underneath both
sit within a factor of about $2.5$ of them, which is as close as an order-of-magnitude
argument promises.

The left branch is the counter-intuitive one, and the reason to have drawn this at all:
**making $h$ smaller eventually makes the answer worse.** At $h = 10^{-14}$,
$f(x+h)$ and $f(x-h)$ agree in almost every significant digit, the subtraction cancels
them, and what survives is rounding noise divided by a tiny number.

The practical conclusions for the rest of the week:

- Central differences beat forward ones by about three orders of magnitude at their
  respective best, for roughly twice the cost ($2n$ evaluations against $n+1$). Use them
  for checking.
- At the default $h = 10^{-6}$ a central difference lands near $10^{-10}$ relative — so
  the $10^{-5}$ tolerance of `check_gradient` is generous by five orders of magnitude.
  When it fails, the formula is wrong; the tolerance is not the problem.
- This is also why the Week 1 autodiff module matters. It is **exact**: no $h$, no
  U-curve, no floor. Finite differences catch sign and scale errors; autodiff catches
  everything, to machine precision.

---

## 6. Now break the gradient checker

Every figure so far has shown something working. The notebooks of this week all end the
same way, and it is not decoration: **a method you have only seen succeed is a method you
do not yet understand.**

`check_gradient` is the tool the whole week is verified with. So: can it be fooled?

Take $f(x) = \sum_i x_i^3$, whose gradient is $3x_i^2$, and a wrong gradient $2x_i^2$ —
a plausible slip, the kind of exponent-versus-coefficient error that actually happens.

In [ ]:
f = lambda u: float(np.sum(u**3))                   # noqa: E731
correct = lambda u: 3 * u**2                        # noqa: E731
wrong = lambda u: 2 * u**2                          # noqa: E731

points = [
    ("x = (0, 0)      every term vanishes", np.zeros(2)),
    ("x = (0.5, -1.5) an ordinary point  ", np.array([0.5, -1.5])),
]

for label, x in points:
    for name, g in [("correct", correct), ("wrong  ", wrong)]:
        try:
            rel = check_gradient(f, g, x)
            verdict = f"PASS   relative error {rel:.3e}"
        except AssertionError as err:
            verdict = f"FAIL   {err}"
        print(f"{label}   {name}   {verdict}")
    print()

**The wrong gradient passes at the origin.** $3 \cdot 0^2$ and $2 \cdot 0^2$ are both
zero, finite differences agree with both, and the check reports success.

This is not a defect in `check_gradient` — it is the nature of testing at a point. The
lesson is about how you *use* it:

- **Never check at a convenient point.** Zero, the origin, $w = 0$, a point of symmetry:
  these are exactly where wrong formulas survive. Check at a point drawn from
  `rng.normal`, and check at more than one.
- A gradient check that has never failed has told you nothing. Before trusting it on new
  code, break the gradient on purpose and confirm it screams.
- $w = 0$ is a particularly dangerous habit here, because it is the natural starting
  point for every optimizer you will write this week — so it is the point you are most
  tempted to test at.

The contract test `tests/contracts/test_objective_contract.py` applies this to every
`IObjective` you write, at random points, which is why it catches things your own tests
miss.

In [ ]:
# The fix: several random points, and report the worst one.
def check_everywhere(f, grad, n, trials=20, rng=rng):
    worst = 0.0
    for _ in range(trials):
        x = rng.normal(size=n)
        gn = numerical_gradient(f, x)
        worst = max(worst, float(np.linalg.norm(grad(x) - gn) / max(1.0, np.linalg.norm(gn))))
    return worst


print(f"correct gradient, worst of 20 random points : {check_everywhere(f, correct, 2):.3e}")
print(f"wrong   gradient, worst of 20 random points : {check_everywhere(f, wrong, 2):.3e}")

# And the same discipline on the objects you actually built today.
print()
for name, obj, x in [
    ("Rosenbrock       ", Rosenbrock(), rng.normal(size=4)),
    ("Quadratic k=1e3  ", Quadratic.ill_conditioned(5, 1e3), rng.normal(size=5)),
    ("linear regression", linear_regression(rng.normal(size=(30, 4)), rng.normal(size=30)),
     rng.normal(size=4)),
    ("logistic regr.   ", logistic_regression(rng.normal(size=(30, 4)),
                                              (rng.random(30) < 0.5).astype(float)),
     rng.normal(size=4)),
]:
    print(f"{name}  check_gradient = {check_gradient(obj.value, obj.gradient, x):.3e}")

---

## Checkpoint

You should now be able to say, without looking anything up:

1. Why squared error is the *right* loss for Gaussian noise — and what "right" means
   there. (Figure 1.)
2. Why the logistic loss is gentler on outliers than squared error, in terms of
   $\partial\varphi/\partial z$. (Figure 2.)
3. Why `log(1 + exp(z))` must not be written that way, and what replaces it.
4. What $\kappa$ measures, how it shows up in the level sets — $\sqrt{\kappa}$, not
   $\kappa$ — and what $(\kappa-1)/(\kappa+1)$ predicts. (Figure 3.)
5. Why standardizing features is an optimization decision. (Figure 4.)
6. Why a smaller $h$ is not a better $h$. (Figure 5.)
7. One way to fool `check_gradient`, and the habit that prevents it.

### Tomorrow

Day 2 builds the loop that walks down these level sets, and the first figure of
notebook 2 is the zigzag that Figure 3 above already predicts: with $\kappa = 100$ the
steepest-descent direction is nearly $80°$ away from the direction of the minimum, so the
iterates will bounce across the valley rather than travel along it. You will measure the
rate and compare it against $(\kappa-1)/(\kappa+1)$.

Keep this notebook. Figures 3 and 5 are referenced again on days 2 and 4.